# 1.消息类型

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- `SystemMessage`：`role`是`system`，代表系统消息，用于设定模型角色和交互背景
- `HumanMessage`：`role`是`user`，代表用户输入的消息
- `AIMessage`：`role`是`assistant`，代表LLM生成的响应，包含：文本、工具调用、元数据
- `ToolMessage`：`role`是`tool`，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息

In [ ]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """获取指定位置的天气信息"""
    return f"Current weather in {location} is sunny"

# 创建 Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用 Agent，发送消息
response = agent.invoke({
    "messages": [
        # {"role": "system", "content": "你是一个热心的AI助手。"},
        # {"role": "user", "content": "你好，我是heima"},
        # {"role": "system", "content": "你好，heima，很高心认识你。"},
        # {"role": "user", "content": "北京天气如何？"},
        SystemMessage(content="你是一个热心的AI助手。"),
        HumanMessage(content="你好，我是heima"),
        AIMessage(content="你好，heima，很高心认识你。"),
        HumanMessage(content="北京天气如何？")
    ]
})

print(response)

In [ ]:
for message in response["messages"]:
    message.pretty_print()  # 使用 pretty_print 方法以更友好的格式打印消息

# 2.多模态消息

![image-20260415172208684](https://cdn.jsdelivr.net/gh/XVSHIFU/Picture-bed@img/image-20260415172208684.png)

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...
-
我们以qwen3.5-plus为例，演示向模型发送图片消息

参考LangChain官方文档：https://docs.langchain.com/oss/python/langchain/messages#multimodal

Image input:

```python
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {
            "type": "image",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "image/jpeg",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "file_id": "file-abc123"},
    ]
}
```

## 2.1 在线图片

In [ ]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model = "qwen3.5-plus",
    model_provider = "openai",
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
    api_key = os.getenv("DASHSCOPE_API_KEY")
)

In [ ]:
# 创建 Agent
agent = create_agent(model=model)

In [ ]:
# 准备多模态消息
# message = {
#     "role": "user",
#     "content": [
#         {"type": "text", "text": "Describe the content of this image."},
#         {"type": "image", "url": "https://img95.699pic.com/element/40218/5277.png_300.png"},
#     ]
# }

message = HumanMessage([
    {"type": "text", "text": "描述以下这张图片的内容"},
    {"type": "image", "url": "https://img95.699pic.com/element/40218/5277.png_300.png"},
])

In [ ]:
# 调用 Agent，发送多模态消息
stream = agent.stream(
    {"messages": [message]},
    stream_mode = "messages"
)

for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

## 2.2 本地图片数据

In [ ]:
!uv add ipywidgets

创建一个上传组件，用于模拟图片上传

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept="*", multipart=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
# 读取图片，转为 base64 字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图 -> bytes
img_bytes = bytes(content_mv)

# base64 编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
# 组织多模态消息
multimodal_question = HumanMessage([
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg"  # 根据实际图片类型设置
    },
    {"type": "text", "text": "描述这张图片的内容"},
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode = "messages"
):
    if chunk.content:
        print(chunk.content, end="", flush=True)